# Merge BioMistral adapter and quantize to GGUF (Q4_K_M)
Settings before running: **Accelerator = None** (CPU is enough), **Internet = On**.
Intermediate files go to `/kaggle/temp` (large, not saved). Only the final file goes to `/kaggle/working`.

## 1. Setup

In [ ]:
!pip install -q peft accelerate huggingface_hub sentencepiece protobuf gguf

import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"   # keep the 14 GB download off /kaggle/working

## 2. Find your adapter folder
Copy the folder that contains `adapter_config.json`.

In [ ]:
!find /kaggle/input -name "adapter_config.json"

## 3. Merge the adapter into the base model (CPU)

In [ ]:
import gc
import torch
from huggingface_hub import hf_hub_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "BioMistral/BioMistral-7B"
ADAPTER_DIR = "/kaggle/input/CHANGE-ME/finetuned-model/final"   # <-- from step 2
MERGED_DIR = "/kaggle/temp/merged"

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(MERGED_DIR)
try:
    hf_hub_download(BASE_MODEL, "tokenizer.model", local_dir=MERGED_DIR)
except Exception as err:
    print("Skipped tokenizer.model:", err)

del model, base
gc.collect()

In [ ]:
# Free disk: the base model download is no longer needed
!rm -rf /kaggle/temp/hf

## 4. Convert to GGUF (fp16)

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/temp/llama.cpp
!python /kaggle/temp/llama.cpp/convert_hf_to_gguf.py /kaggle/temp/merged --outfile /kaggle/temp/biomistral-f16.gguf --outtype f16
!rm -rf /kaggle/temp/merged

## 5. Build the quantizer and make the 4-bit file

In [ ]:
!cd /kaggle/temp/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4

In [ ]:
!/kaggle/temp/llama.cpp/build/bin/llama-quantize /kaggle/temp/biomistral-f16.gguf /kaggle/working/biomistral-med-q4_k_m.gguf Q4_K_M
!ls -lh /kaggle/working